Instalamos ambas paqueterías para poder usarlas (optuna y mlflow)

In [ ]:
!pip install optuna

In [ ]:
!pip install mlflow optuna

In [ ]:
!pip install -q dagshub mlflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.0/262.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.2 MB/s eta 0:00:00


conectamos a mlflow, luego creamos un experimento y abrimos una ventana en nuestro servidor local para ver la interfaz del experimento

In [ ]:
import mlflow
import dagshub
dagshub.init(repo_owner='JulioDolores', repo_name='redes-neuronales', mlflow=True)

# Configurar tracking URI de MLflow
mlflow.set_tracking_uri('https://dagshub.com/JulioDolores/redes-neuronales.mlflow')
mlflow.set_experiment("MNIST_Dense_Optuna_Experiment")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4960539c-765a-465f-a193-a07f906e0038&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1fa0c06587cee3a05a48fd6be4c951b601766c5bdd9c3e444032eaac027afe68




Accessing as JulioDolores

Initialized MLflow to track repo "JulioDolores/redes-neuronales"

Repository JulioDolores/redes-neuronales initialized!

<Experiment: artifact_location='mlflow-artifacts:/9476024894044b5f8448bdd60cf36a40', creation_time=1770939919499, experiment_id='0', last_update_time=1770939919499, lifecycle_stage='active', name='MNIST_Dense_Optuna_Experiment', tags={}>

In [ ]:
import optuna
import tensorflow as tf
#import mlflow
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.utils import to_categorical

Preparamos los datos que se van a ocupar

In [ ]:
# 1. Cargamos el dataset MNIST
from sklearn.model_selection import train_test_split

(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

x_train, x_test = x_train/255., x_test/255.

y_train, y_test = to_categorical(y_train), to_categorical(y_test)

# Dividimos el conjunto de entrenamiento para crear un conjunto de validación
x_train_split, x_val, y_train_split, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Definimos la función objetivo

In [ ]:
# Definimos la función create_model

def create_model(trial):
    model = models.Sequential()
    model.add(layers.Flatten(input_shape = (28, 28)))
    n_layers = trial.suggest_int("n_layers", 1, 5)

    # Optuna sugiere: Número de unidades por capa

    for i in range(n_layers):
        n_units = trial.suggest_int(f"n_units_l{i}", 32, 128)
        # Función de activación
        activation = trial.suggest_categorical(
            f'activation_l{i}',
            ['relu', 'tanh', 'sigmoid', 'elu', 'selu', 'softplus']
        )
        model.add(layers.Dense(n_units, activation=activation))
    model.add(layers.Dense(10, activation = 'softmax'))

    # Optuna sugiere: Valor de learning rate
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log = True)
    # Optuna sugiere: Optimizador
    optimizer_options = ["sgd", "adam", "rmsprop",'adamax']

    optimizer_selected = trial.suggest_categorical("optimizer", optimizer_options)

    if optimizer_selected == "adam":

        optimizer = tf.keras.optimizers.Adam(learning_rate = lr)

    elif optimizer_selected == "sgd":

        optimizer = tf.keras.optimizers.SGD(learning_rate = lr)

    elif optimizer_selected == "rmsprop":

        optimizer = tf.keras.optimizers.RMSprop(learning_rate = lr)

    elif optimizer_selected == "adamax":

        optimizer = tf.keras.optimizers.Adamax(learning_rate=lr)

    # Compilamos el modelo

    model.compile(optimizer = optimizer,
                  loss = "categorical_crossentropy",
                  metrics = ["accuracy"])

    return model

In [ ]:
def objective(trial):
    # Start an MLflow run for each Optuna trial
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        # Crear modelo - esta llamada llenará trial.params con los hiperparámetros sugeridos
        modelo = create_model(trial)

        # Registrar cada hiperparámetro de Optuna individualmente a MLflow
        for key, value in trial.params.items():
            mlflow.log_param(key, value)

        # Early Stopping para mejor entrenamiento
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        # Entrenar modelo
        history = modelo.fit(
            x_train_split, y_train_split, # Usamos el conjunto de entrenamiento dividido
            epochs=20,  # Aumentado para mejor convergencia
            batch_size=64,
            validation_data=(x_val, y_val), # Usamos el conjunto de validación definido
            callbacks=[early_stopping],
            verbose=0
        )

        loss, accuracy = modelo.evaluate(x_test, y_test, verbose = 0)

        # Registrar métricas en MLflow
        mlflow.log_metric("val_loss", loss)
        mlflow.log_metric("val_accuracy", accuracy)
        mlflow.log_metric("best_val_accuracy", max(history.history['val_accuracy']))

        return loss

entrenamos las redes con diferentes optimizadores, funciones de activación.

Los resultados empiezan a generarse en experimentos de mlflow

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

study = optuna.create_study(direction = "minimize")

study.optimize(objective, n_trials = 15)

[I 2026-02-13 00:55:41,099] A new study created in memory with name: no-name-adf8787f-90f5-4294-83c7-fbd8e9efae40


🏃 View run agreeable-pug-632 at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0/runs/035c6d16675446bdbd26fc009f2cc18a
🧪 View experiment at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.

[I 2026-02-13 00:56:03,789] Trial 0 finished with value: 0.11751611530780792 and parameters: {'n_layers': 2, 'n_units_l0': 46, 'activation_l0': 'selu', 'n_units_l1': 48, 'activation_l1': 'softplus', 'lr': 0.00372962888149098, 'optimizer': 'rmsprop'}. Best is trial 0 with value: 0.11751611530780792.


🏃 View run optuna_trial_0 at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0/runs/2ac418741bec44218508e3d9e8e55d8f
🧪 View experiment at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0


[I 2026-02-13 00:56:31,052] Trial 1 finished with value: 0.0906616672873497 and parameters: {'n_layers': 2, 'n_units_l0': 80, 'activation_l0': 'sigmoid', 'n_units_l1': 32, 'activation_l1': 'elu', 'lr': 0.016723560488556822, 'optimizer': 'adamax'}. Best is trial 1 with value: 0.0906616672873497.


...

🏃 View run optuna_trial_12 at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0/runs/c1b500db57e44720b994472ff7a8b4a8
🧪 View experiment at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0


[I 2026-02-13 01:08:17,381] Trial 13 finished with value: 0.10556423664093018 and parameters: {'n_layers': 3, 'n_units_l0': 108, 'activation_l0': 'tanh', 'n_units_l1': 122, 'activation_l1': 'sigmoid', 'n_units_l2': 123, 'activation_l2': 'softplus', 'lr': 0.00012753407856375682, 'optimizer': 'adam'}. Best is trial 5 with value: 0.08335885405540466.


🏃 View run optuna_trial_13 at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0/runs/a41d73af72994c4aa413a22d4acb7bdf
🧪 View experiment at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0


[I 2026-02-13 01:09:32,912] Trial 14 finished with value: 0.2467219978570938 and parameters: {'n_layers': 5, 'n_units_l0': 119, 'activation_l0': 'relu', 'n_units_l1': 98, 'activation_l1': 'softplus', 'n_units_l2': 33, 'activation_l2': 'elu', 'n_units_l3': 34, 'activation_l3': 'relu', 'n_units_l4': 124, 'activation_l4': 'elu', 'lr': 1.0479795358831092e-05, 'optimizer': 'adam'}. Best is trial 5 with value: 0.08335885405540466.


🏃 View run optuna_trial_14 at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0/runs/38f292cc96df4361bbaf4d61b7407faa
🧪 View experiment at: https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0


Obtenemos el mejor intento

In [ ]:
print("Número de pruebas terminadas: ", len(study.trials))

trial = study.best_trial

print("Mejor intento: ", trial)


print("Valor: ", trial.value)
print("Hiperparámetros: ", trial.params)

Número de pruebas terminadas:  15
Mejor intento:  FrozenTrial(number=5, state=<TrialState.COMPLETE: 1>, values=[0.08335885405540466], datetime_start=datetime.datetime(2026, 2, 13, 0, 59, 17, 449443), datetime_complete=datetime.datetime(2026, 2, 13, 1, 0, 10, 195379), params={'n_layers': 1, 'n_units_l0': 116, 'activation_l0': 'selu', 'lr': 0.0005624044460412629, 'optimizer': 'rmsprop'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_layers': IntDistribution(high=5, log=False, low=1, step=1), 'n_units_l0': IntDistribution(high=128, log=False, low=32, step=1), 'activation_l0': CategoricalDistribution(choices=('relu', 'tanh', 'sigmoid', 'elu', 'selu', 'softplus')), 'lr': FloatDistribution(high=0.1, log=True, low=1e-05, step=None), 'optimizer': CategoricalDistribution(choices=('sgd', 'adam', 'rmsprop', 'adamax'))}, trial_id=5, value=None)
Valor:  0.08335885405540466
Hiperparámetros:  {'n_layers': 1, 'n_units_l0': 116, 'activation_l0': 'selu', 'lr': 0.0005624044460

In [ ]:
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_slice
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank

In [ ]:
plot_optimization_history(study)

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_slice(study)

In [ ]:
plot_param_importances(study)

In [ ]:
plot_rank(study)

Ponemos los datos de la mejor red que se entrenó

In [ ]:
mejores_par = {
    'n_layers': 1,
    'n_units_l0': 116,
    'activation_l0': 'selu',
    'optimizer': 'rmsprop',
    'lr': 0.0005624044460412629,
    'batch_size': 64
}

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.datasets import mnist

Ponemos la función para crear el modelo con diferentes regularizaciones

In [ ]:
def create_model(regularization_type=None, reg_strength=0.01, dropout_rate=0.0):
    model = keras.Sequential()
    model.add(layers.Input(shape=(28, 28)))
    model.add(layers.Flatten())

    kernel_regularizer_instance = None
    if regularization_type == regularizers.l1:
        kernel_regularizer_instance = regularizers.l1(reg_strength)
    elif regularization_type == regularizers.l2:
        kernel_regularizer_instance = regularizers.l2(reg_strength)
    elif regularization_type == regularizers.l1_l2:
        kernel_regularizer_instance = regularizers.l1_l2(l1=reg_strength, l2=reg_strength)
    model.add(layers.Dense(116, activation='selu', kernel_regularizer=kernel_regularizer_instance))

    if dropout_rate > 0:
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation='softmax'))

    model.compile(optimizer=mejores_par['optimizer'], loss='categorical_crossentropy', metrics=['accuracy'])
    return model

Entrenamiento con regularización $L_1$

In [ ]:
model_l1 = create_model(regularization_type=regularizers.l1(0.01))
history_l1 = model_l1.fit(x_train_split, y_train_split, epochs=20, batch_size=64, validation_data=(x_val, y_val))

mlflow.start_run(run_name="L1_Regularizacion")
mlflow.log_param("Regularization", "L1")
mlflow.log_metric("Train Accuracy", history_l1.history['accuracy'][-1])
mlflow.log_metric("Validation Accuracy", history_l1.history['val_accuracy'][-1])
mlflow.end_run()


Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8534 - loss: 0.5119 - val_accuracy: 0.9242 - val_loss: 0.2580
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9332 - loss: 0.2362 - val_accuracy: 0.9432 - val_loss: 0.2040
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9489 - loss: 0.1681 - val_accuracy: 0.9543 - val_loss: 0.1556
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9612 - loss: 0.1342 - val_accuracy: 0.9622 - val_loss: 0.1313
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9708 - loss: 0.1058 - val_accuracy: 0.9647 - val_loss: 0.1183
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9741 - loss: 0.0900 - val_accuracy: 0.9664 - val_loss: 0.1158
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9782 - loss: 0.0743 - val_accuracy: 0.9684 - val_loss: 0.1070
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9817 - loss: 0.0661 - val_accuracy: 0.

Entrenamiento con regularización $L_2$

In [ ]:
model_l2 = create_model(regularization_type=regularizers.l2)
history_l2 = model_l2.fit(x_train_split, y_train_split, epochs=20, batch_size=64, validation_data=(x_val, y_val))

mlflow.start_run(run_name="L2_Regularizacion")
mlflow.log_param("Regularization", "L2")
mlflow.log_metric("Train Accuracy", history_l1.history['accuracy'][-1])
mlflow.log_metric("Validation Accuracy", history_l1.history['val_accuracy'][-1])
mlflow.end_run()

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8428 - loss: 1.1571 - val_accuracy: 0.8925 - val_loss: 0.4717
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8973 - loss: 0.4498 - val_accuracy: 0.8988 - val_loss: 0.4445
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9016 - loss: 0.4274 - val_accuracy: 0.9046 - val_loss: 0.4281
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9026 - loss: 0.4235 - val_accuracy: 0.9134 - val_loss: 0.3985
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9047 - loss: 0.4148 - val_accuracy: 0.9082 - val_loss: 0.3993
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9052 - loss: 0.4061 - val_accuracy: 0.9158 - val_loss: 0.3827
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9107 - loss: 0.4021 - val_accuracy: 0.8990 - val_loss: 0.4359
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9099 - loss: 0.3965 - val_accuracy: 0.

Entrenamiento con regularización $L_1L_2$

In [ ]:
model_l1_l2 = create_model(regularization_type=regularizers.l1_l2)
history_l1_l2 = model_l1_l2.fit(x_train_split, y_train_split, epochs=20, batch_size=64, validation_data=(x_val, y_val))

mlflow.start_run(run_name="L1_L2_Regularizacion")
mlflow.log_param("Regularization", "L1_L2")
mlflow.log_metric("Train Accuracy", history_l1.history['accuracy'][-1])
mlflow.log_metric("Validation Accuracy", history_l1.history['val_accuracy'][-1])
mlflow.end_run()

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7341 - loss: 7.0466 - val_accuracy: 0.8096 - val_loss: 1.6098
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8031 - loss: 1.6048 - val_accuracy: 0.7297 - val_loss: 1.7395
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8180 - loss: 1.4952 - val_accuracy: 0.8545 - val_loss: 1.3584
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8293 - loss: 1.4236 - val_accuracy: 0.8298 - val_loss: 1.4331
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8334 - loss: 1.3801 - val_accuracy: 0.8341 - val_loss: 1.3497
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8425 - loss: 1.3452 - val_accuracy: 0.8624 - val_loss: 1.2523
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8457 - loss: 1.3130 - val_accuracy: 0.8181 - val_loss: 1.4390
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8525 - loss: 1.2801 - val_accuracy: 0

Entrenamiento con dropout

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

model_dropout = create_model(dropout_rate=0.5)
history_dropout = model_dropout.fit(x_train_split, y_train_split, epochs=20, batch_size=64, validation_data=(x_val, y_val))

with mlflow.start_run(run_name="Dropout_Model"):
    mlflow.log_param("Regularization_Type", "Dropout")
    mlflow.log_param("dropout_rate", 0.5)
    mlflow.log_metric("Train_Accuracy", history_dropout.history['accuracy'][-1])
    mlflow.log_metric("Validation_Accuracy", history_dropout.history['val_accuracy'][-1])

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8071 - loss: 0.6387 - val_accuracy: 0.9161 - val_loss: 0.2946
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8967 - loss: 0.3623 - val_accuracy: 0.9202 - val_loss: 0.2784
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9077 - loss: 0.3275 - val_accuracy: 0.9264 - val_loss: 0.2625
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9111 - loss: 0.3069 - val_accuracy: 0.9289 - val_loss: 0.2475
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9182 - loss: 0.2834 - val_accuracy: 0.9372 - val_loss: 0.2261
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9193 - loss: 0.2808 - val_accuracy: 0.9345 - val_loss: 0.2235
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9240 - loss: 0.2593 - val_accuracy: 0.9371 - val_loss: 0.2176
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9263 - loss: 0.2544 - val_accuracy: 0.

Entrenamiento con dropout y regularización $L_1L_2$

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

model_dropout_l1_l2 = create_model(regularization_type=regularizers.l1_l2, dropout_rate=0.5)
history_dropout_l1_l2 = model_dropout_l1_l2.fit(x_train_split, y_train_split, epochs=20, batch_size=64, validation_data=(x_val, y_val))

mlflow.start_run(run_name="Dropout_L1_L2_Regularizacion")
mlflow.log_param("Regularization_Type", "Dropout + L1_L2")
mlflow.log_param("Regularization", "L1_L2")
mlflow.log_metric("Train Accuracy", history_l1.history['accuracy'][-1])
mlflow.log_metric("Validation Accuracy", history_l1.history['val_accuracy'][-1])
mlflow.end_run()

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7046 - loss: 7.4451 - val_accuracy: 0.7562 - val_loss: 1.8727
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7683 - loss: 1.7910 - val_accuracy: 0.7953 - val_loss: 1.7727
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7787 - loss: 1.7208 - val_accuracy: 0.8391 - val_loss: 1.5166
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7765 - loss: 1.7002 - val_accuracy: 0.7906 - val_loss: 1.6143
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7826 - loss: 1.6725 - val_accuracy: 0.8587 - val_loss: 1.4951
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7850 - loss: 1.6573 - val_accuracy: 0.8268 - val_loss: 1.5561
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7892 - loss: 1.6449 - val_accuracy: 0.8353 - val_loss: 1.4835
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7866 - loss: 1.6418 - val_accuracy: 0.

Los resultados de la red sin utlizar optimizador es  

| Metric | Value |
|--------|-------|
| $val\_accuracy$ | $0.9760000109672546$ |
| $best\_val\_accuracy$ | $0.9743333458900452$ |
| $val\_loss$ | $0.08335885405540466$ |

##Conclusiones

1. La regularización L1 ha mantenido un rendimiento de validación casi idéntico al modelo base. Pero, hay muy poca diferencia entre entrenamiento y validación sigue siendo pequeña, lo que indica que el sobreajuste no era un problema grave para empezar. El ligero aumento en val_loss podría ser una señal de que la regularización es un poco agresiva, pero el impacto es mínimo.
2. La regularización L2 y Dropout, con la fuerza que se utilizaron, es demasiado porque elimina el sobreajuste, pero empeora la capacidad predictiva.
3. Con L1L2 y dropout-L1L2 es totalemente excesivo, empeora muuchisimo la capacidad predictiva

Link para acceder a los experimentos

https://dagshub.com/JulioDolores/redes-neuronales.mlflow/#/experiments/0/runs?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false&startTime=ALL&lifecycleFilter=Active&modelVersionFilter=All+Runs&datasetsFilter=W10%3D